# sklearn Tutorial — Papermill on Colab

End-to-end sklearn classification with structured outputs (PNGs, logs, metrics CSV).

**Model:** SGDClassifier with `partial_fit` (real training curves) + RandomForest comparison.
**Dataset:** sklearn digits (8×8 images, 10 classes, 1797 samples).
**Runtime:** ~2-3 minutes on Colab CPU.

This notebook is designed for papermill parameter injection — edit the `parameters` cell
defaults and run locally, or override via CLI:
```bash
papermill tutorial.ipynb output.ipynb -p n_epochs 30 -p alpha 0.0005
```

In [ ]:
# Papermill parameters — override via CLI: papermill ... -p n_epochs 30 -p alpha 0.0005
n_epochs = 10
alpha = 0.0001
test_size = 0.2
random_state = 42

In [ ]:
# Install + imports + output dir setup
import os, sys, time, json, csv
from datetime import datetime
from pathlib import Path

# Ensure log_utils / plot_utils are importable from /content/
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

from log_utils import Logger, MetricsCSV, SummaryJSON, setup_output_dirs
from plot_utils import plot_loss_acc

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, log_loss)

OUT_DIR = '/content/tutorial-output'
setup_output_dirs(OUT_DIR)

logger = Logger(f'{OUT_DIR}/logs/train.log')
logger.log(f'Starting sklearn tutorial — n_epochs={n_epochs} alpha={alpha}')
logger.log(f'Out dir: {OUT_DIR}')

print(f'Parameters: n_epochs={n_epochs}, alpha={alpha}, test_size={test_size}, random_state={random_state}')

In [ ]:
# Load and prepare data
t0 = time.time()

digits = load_digits()
X, y = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

logger.log(
    f'Data: {X.shape[0]} samples, {X.shape[1]} features, {len(np.unique(y))} classes | '
    f'train={X_train.shape[0]} test={X_test.shape[0]}'
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Train SGDClassifier with partial_fit — produces real per-epoch loss curves
classes = np.unique(y_train)

sgd = SGDClassifier(
    loss='log_loss',
    alpha=alpha,
    max_iter=1,  # we control epochs manually
    random_state=random_state,
    warm_start=True,
)

metrics_history = []
t_start = time.time()

for epoch in range(1, n_epochs + 1):
    sgd.partial_fit(X_train, y_train, classes=classes)

    train_acc = accuracy_score(y_train, sgd.predict(X_train))
    test_acc = accuracy_score(y_test, sgd.predict(X_test))
    train_loss = log_loss(y_train, sgd.predict_proba(X_train))
    test_loss = log_loss(y_test, sgd.predict_proba(X_test))
    elapsed = time.time() - t_start

    metrics_history.append({
        'epoch': epoch,
        'train_loss': round(train_loss, 6),
        'train_acc': round(train_acc, 6),
        'test_loss': round(test_loss, 6),
        'test_acc': round(test_acc, 6),
        'elapsed_s': round(elapsed, 1),
    })

    logger.log(
        f'Ep {epoch:3d}/{n_epochs} | '
        f'train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | '
        f'test_loss={test_loss:.4f} | test_acc={test_acc:.4f} | '
        f'elapsed={elapsed:.0f}s'
    )

logger.log(f'SGD training done — {n_epochs} epochs in {time.time()-t_start:.0f}s')
print(f'SGD final: train_acc={metrics_history[-1]["train_acc"]:.4f}, test_acc={metrics_history[-1]["test_acc"]:.4f}')

In [ ]:
# Train RandomForest for comparison
t0 = time.time()

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=random_state,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

rf_train_acc = accuracy_score(y_train, rf.predict(X_train))
rf_test_acc = accuracy_score(y_test, rf.predict(X_test))

logger.log(
    f'RandomForest: train_acc={rf_train_acc:.4f} test_acc={rf_test_acc:.4f} '
    f'time={time.time()-t0:.0f}s'
)
print(f'RF train_acc={rf_train_acc:.4f} test_acc={rf_test_acc:.4f}')

In [ ]:
# Generate 4-panel training curves with plot_utils
# plot_loss_acc expects a dict: {batch_losses, eval_losses, eval_accs, lr_values}
plot_metrics = {
    "batch_losses": [m['train_loss'] for m in metrics_history],
    "eval_losses": [(m['epoch'], m['test_loss']) for m in metrics_history],
    "eval_accs": [(m['epoch'], m['test_acc']) for m in metrics_history],
}
plot_loss_acc(
    plot_metrics,
    f'{OUT_DIR}/pngs/training_curves.png',
    title=f'SGDClassifier — sklearn Digits (alpha={alpha})',
    size_label='small',
)
logger.log(f'PNG saved: {OUT_DIR}/pngs/training_curves.png')
print('training_curves.png written')

In [ ]:
# Extra diagnostic plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix for best model (SGD)
cm = confusion_matrix(y_test, sgd.predict(X_test))
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_title(f'SGD Confusion Matrix (test_acc={metrics_history[-1]["test_acc"]:.3f})')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[0].text(j, i, cm[i, j], ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=axes[0])

# Learning curve (sklearn built-in)
train_sizes, train_scores, test_scores = learning_curve(
    RandomForestClassifier(n_estimators=50, random_state=random_state, n_jobs=-1),
    X_train, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy',
)
train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)
axes[1].plot(train_sizes, train_mean, 'o-', label='Train', color='#2563eb')
axes[1].plot(train_sizes, test_mean, 'o-', label='CV Test', color='#ea580c')
axes[1].fill_between(train_sizes, train_mean - train_scores.std(axis=1),
                     train_mean + train_scores.std(axis=1), alpha=0.15, color='#2563eb')
axes[1].fill_between(train_sizes, test_mean - test_scores.std(axis=1),
                     test_mean + test_scores.std(axis=1), alpha=0.15, color='#ea580c')
axes[1].set_title('RandomForest Learning Curve')
axes[1].set_xlabel('Training samples')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/pngs/diagnostics.png', dpi=100, bbox_inches='tight')
plt.close()
logger.log(f'PNG saved: {OUT_DIR}/pngs/diagnostics.png')
print('diagnostics.png written')

In [ ]:
# Write metrics CSV
csv = MetricsCSV(
    f'{OUT_DIR}/metrics.csv',
    ['epoch', 'train_loss', 'train_acc', 'test_loss', 'test_acc', 'elapsed_s'],
)
for row in metrics_history:
    csv.write_row(**row)
logger.log(f'CSV saved: {OUT_DIR}/metrics.csv ({len(metrics_history)} rows)')

In [ ]:
# Write summary JSON
summary = SummaryJSON(f'{OUT_DIR}/summary.json')
summary.write({
    'model': 'SGDClassifier',
    'dataset': 'sklearn-digits',
    'n_epochs': n_epochs,
    'alpha': alpha,
    'test_size': test_size,
    'sgd_train_acc': metrics_history[-1]['train_acc'],
    'sgd_test_acc': metrics_history[-1]['test_acc'],
    'sgd_train_loss': metrics_history[-1]['train_loss'],
    'sgd_test_loss': metrics_history[-1]['test_loss'],
    'rf_train_acc': round(rf_train_acc, 6),
    'rf_test_acc': round(rf_test_acc, 6),
    'total_time_s': round(time.time() - t_start, 1),
    'n_train': int(X_train.shape[0]),
    'n_test': int(X_test.shape[0]),
})
logger.log(f'Summary saved: {OUT_DIR}/summary.json')
logger.log('Done.')
print('All outputs written. Done.')

## Results

Output files on the VM:
- `logs/train.log` — timestamped per-epoch log
- `metrics.csv` — one row per epoch, all scalar metrics
- `pngs/training_curves.png` — 4-panel training curves (loss, accuracy, LR, distribution)
- `pngs/diagnostics.png` — confusion matrix + learning curve
- `summary.json` — final run metadata

Download with:
```bash
tar -czf tutorial-output.tar.gz -C /content tutorial-output
# then: colab download /content/tutorial-output.tar.gz ./tutorial-output.tar.gz
```